# Transactions

### Importing required libraries

In [ ]:
import sqlite3
import pandas as pd


### Connecting to SQLite database

In [ ]:
%load_ext sql

In [ ]:
%sql sqlite:///mydb.db

### Creating tables

In [ ]:
%%sql


CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT
);


In [ ]:
%%sql

CREATE TABLE organizations (
    organization_id INTEGER PRIMARY KEY,
    organization_name TEXT,
    category_id INTEGER,
    organization TEXT,
    location TEXT,

    FOREIGN KEY (category_id) REFERENCES categories(category_id)
);

In [ ]:
%%sql

CREATE TABLE operation_status (
    operationStatusId INTEGER PRIMARY KEY,
    operationName TEXT,
    Description_ TEXT
);

In [ ]:
%%sql 

CREATE TABLE transactions (
    Id INTEGER PRIMARY KEY,
    OrgId INTEGER,
    TransactionTypeId INTEGER,
    Transaction_Date TEXT,
    OperationStatusId INTEGER,
    CardlNumber TEXT,
    PersonId INTEGER,
    
    FOREIGN KEY (OrgId) REFERENCES organizations(organization_id),
    FOREIGN KEY (OperationStatusId) REFERENCES operation_status(operationStatusId)
);

### Importing tables

In [ ]:
import pandas as pd

org_df = pd.read_excel("Organizations1.xlsx")
cat_df = pd.read_excel("Categories1.xlsx")
status_df = pd.read_excel("OperationStatus1.xlsx")
trans_df = pd.read_excel("Intern_dataset(in) son1.xlsx")

In [ ]:
cat_df.to_sql("categories", conn, if_exists="append", index=False)

In [ ]:
status_df.to_sql("operation_status", conn, if_exists="append", index=False)

In [ ]:
org_df.to_sql("organizations", conn, if_exists="append", index=False)

In [ ]:
trans_df.to_sql("transactions", conn, if_exists="append", index=False)

In [ ]:
trans_df

### Checking the foreign keys

In [ ]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [ ]:
%%sql
PRAGMA foreign_key_list(transactions);

### How Many Transactions Has Each User Made?

In [ ]:
%%sql

select PersonId,
       COUNT(*)
from transactions
group by PersonId

### How Many Transactions Did Users Perform by Organization Across Years ?

In [ ]:
%%sql

SELECT strftime('%Y', transaction_date) AS year,
       OrgId,
       count(*)
FROM transactions
group by strftime('%Y', transaction_date), OrgId

### Who Are the Top Transactional Users by Organization?

In [ ]:
%%sql


with t1 as 
(select t.*,
       row_number() over (partition by t.organization_name order by t.transaction_count desc) as rownum
from
(select o.organization_name,
       tr.PersonId,
       COUNT(tr.Id) as transaction_count
from transactions tr
left join organizations o on o.organization_id = tr.OrgId
group by OrgId, PersonId) t )
select organization_name,
       PersonId,
       transaction_count
from t1
where t1.rownum=1


### Are There Any Duplicate Card Numbers? If Yes, Which Users Do They Belong To?

In [ ]:
%%sql

select CardlNumber,
       PersonId
from transactions
where CardlNumber in (select CardlNumber
                      from transactions
                      group by CardlNumber
                      having count(distinct PersonId)>1)
group by CardlNumber, PersonId

### Top 10 Organizations by Transaction Count

In [ ]:
%%sql

with t1 as(
select o.organization_name,
       (select count(t.Id)
        from transactions t
        where t.OrgId= o.organization_id) as transaction_count
from organizations o )
select * from t1
order by transaction_count desc
limit 10

### Top 10 Organizations by User Count 

In [ ]:
%%sql

with t1 as(
select o.organization_name,
       (select count(t.PersonId)
        from transactions t
        where t.OrgId= o.organization_id) as user_count
from organizations o )
select * from t1
order by user_count desc
limit 10

### Transaction Distribution by Operation Status

In [ ]:
%%sql

select o.operationName,
       (select count(t.Id)
        from transactions t
        where t.OperationStatusId = o.operationStatusId)
from operation_status o

### Monthly Transaction and User Counts by Organization and Operation Status

In [ ]:
%%sql

select strftime('%m', t.Transaction_Date) AS month,
       o.organization_name,
       t.OperationStatusId,
       count(t.Id) as transaction_count
from transactions t
left join organizations o on o.organization_id = t.OrgId
group by strftime('%m', t.Transaction_Date), o.organization_name, t.OperationStatusId

### Number of Organizations by Category

In [ ]:
%%sql 

select c.category_name,
       count(o.organization_id) as org_count
from categories c
left join organizations o on o.category_id = c.category_id
group by c.category_name

### Transaction Count by Category

In [ ]:
%%sql 

select c.category_name,
       count(t.Id) as transaction_count
from categories c
left join organizations o on o.category_id = c.category_id
left join transactions t on t.OrgId = o.organization_id
group by c.category_name

### Number of Users Performing Transactions by Category

In [ ]:
%%sql 

select c.category_name,
       count(t.PersonId) as user_count
from categories c
left join organizations o on o.category_id = c.category_id
left join transactions t on t.OrgId = o.organization_id
group by c.category_name

### Transaction Count Across Organizations by Category

In [ ]:
%%sql 

select c.category_name,
       o.organization,
       count(t.Id) as transaction_count
from categories c
left join organizations o on o.category_id = c.category_id
left join transactions t on t.OrgId = o.organization_id
group by c.category_name, o.organization

### Yearly Transaction Trends by Category

In [ ]:
%%sql 

select strftime('%Y', t.Transaction_Date) as year,
       c.category_name,
       count(t.Id) as transaction_count
from categories c
left join organizations o on o.category_id = c.category_id
left join transactions t on t.OrgId = o.organization_id
group by strftime('%Y', t.Transaction_Date),c.category_name

### Transaction Count by Location

In [ ]:
%%sql 

select o.location,
       count(t.Id) as transaction_count
from organizations o
left join transactions t on t.OrgId = o.organization_id
group by o.location

### Card vs Cardless Transaction Distribution

In [ ]:
%%sql

select 
       sum(case when CardlNumber is null then 1 else 0 end) as transactions_without_card,
       sum(case when CardlNumber is not null then 1 else 0 end) as transactions_with_card
from transactions

### Top 5 Users with the Highest Number of Cards

In [ ]:
%%sql

select PersonId,
       count(distinct CardlNumber) as card_count
from transactions
where CardlNumber is not null
group by PersonId
order by card_count desc
limit 5

### Average Transaction Count per User

In [ ]:
%%sql

select count(Id)/count(distinct PersonId) as avg_tr_per_user
from transactions

### Top 10 Users with the Highest Daily Transaction Count

In [ ]:
%%sql 

with t1 as(
select t.*,
       row_number() over(partition by t.PersonId order by t.tr_count desc) as rownum
from
(select PersonId,
       count(Id) as tr_count
from transactions
group by PersonId, Transaction_Date) t )

select PersonId, tr_count from t1
where rownum=1
order by tr_count desc 
limit 10

### Which Users Have the Highest Number of Refused Transactions?

In [ ]:
%%sql 

select PersonId,
       sum(case when OperationStatusId=20 then 1 else 0 end) as refused_tr
from transactions
group by PersonId
order by refused_tr desc 
limit 20

### Transaction Counts of Cardless Users by Operation Status

In [ ]:
%%sql

select PersonId,
       count(CardlNumber),
       OperationStatusId,
       count(Id)
from transactions
group by PersonId, OperationStatusId
having count(CardlNumber)=0 

### Refused vs Acknowledged Transactions in Cardless Operations

In [ ]:
%%sql

select 
       sum(case when OperationStatusId=20 then 1 else 0 end) as refused_tr,
       sum(case when OperationStatusId=10 then 1 else 0 end) as acknowledged_tr
from transactions 
where CardlNumber is null

### Which Branches Have the Highest Transaction Counts Within Each Category and Organization?

In [ ]:
%%sql

with t1 as
(select t.*,
       row_number() over (partition by t.category_name order by t.tr_count desc) as rownum
from
(select c.category_name,
       o.organization,
       o.location,
       count(Id) tr_count
from transactions t
left join organizations o on o.organization_id=t.OrgId
left join categories c on c.category_id = o.category_id
group by c.category_name, o.organization, o.location) t ) 

select category_name,
       organization,
       location,
       tr_count
from t1 
where rownum=1

### Transaction Distribution by Category and Operation Status

In [ ]:
%%sql

select c.category_name,
       t.OperationStatusId,
       count(Id) as tr_count
from transactions t
left join organizations o on o.organization_id=t.OrgId
left join categories c on c.category_id=o.category_id
group by c.category_name, t.OperationStatusId

### Transaction Distribution by Hour of the Day

In [ ]:
%%sql

SELECT strftime('%H', Transaction_Date) as Hour,
       count(Id) as tr_count
FROM transactions
group by strftime('%H', Transaction_Date)

### Most Frequently Used Cards

In [ ]:
%%sql

select CardlNumber,
       count(Id) as tr_count
from transactions
where CardlNumber is not null
group by CardlNumber
order by tr_count desc
limit 10

### Which Cards Have the Highest Refusal Rates?

In [ ]:
%%sql

select CardlNumber,
       count(Id) as tr_count,
       sum(case when OperationStatusId=20 then 1 else 0 end) as refused_transactions
from transactions
where CardlNumber is not null
group by CardlNumber
order by refused_transactions desc
limit 10

### Transaction Distribution by Day of the Week

In [ ]:
%%sql 

SELECT 
    day_name,
    COUNT(*) AS transaction_count
FROM (
    SELECT 
        strftime('%w', Transaction_Date) AS day_num,
        CASE strftime('%w', Transaction_Date)
            WHEN '0' THEN 'Sun'
            WHEN '1' THEN 'Mon'
            WHEN '2' THEN 'Tue'
            WHEN '3' THEN 'Wed'
            WHEN '4' THEN 'Thu'
            WHEN '5' THEN 'Fri'
            WHEN '6' THEN 'Sat'
        END AS day_name
    FROM transactions
) t
GROUP BY day_name
ORDER BY day_num;

### Top User by Organization

In [ ]:
%%sql

with t2 as
(select t1.*,
       row_number() over(partition by t1.organization_name order by t1.tr_count desc) as rownum
from
(select o.organization_name,
       t.PersonId,
       count(Id) as tr_count
from transactions t
left join organizations o on o.organization_id=t.OrgId
group by o.organization_name, t.PersonId ) t1 )

select organization_name,
       PersonId,
       tr_count
from t2 
where rownum=1

### Card vs Cardless Transaction Counts by Organization

In [ ]:
%%sql

select o.organization_name,
       sum(case when t.CardlNumber is null then 1 else 0 end) as transactions_without_card,
       sum(case when t.CardlNumber is not null then 1 else 0 end) as transactions_with_card
from transactions t
left join organizations o on o.organization_id=t.OrgId
group by o.organization_name

### Refusal Rate by Organization

In [ ]:
%%sql

select o.organization_name,
       round(sum(case when t.OperationStatusId=20 then 1 else 0 end)*1.0/count(Id),1) as refusal_rate
from transactions t
left join organizations o on o.organization_id=t.OrgId
group by o.organization_name

### User Counts by Operation Status and Transaction Type

In [ ]:
%%sql

select OperationStatusId,
       TransactionTypeId,
       count(distinct PersonId) as user_count
from transactions 
group by OperationStatusId,TransactionTypeId

### Refusal Rate by Category and Transaction Type

In [ ]:
%%sql

select c.category_name,
       t.TransactionTypeId,
       round(sum(case when t.OperationStatusId=20 then 1 else 0 end)*1.0/count(Id),1) as refusal_rate
from transactions t
left join organizations o on o.organization_id=t.OrgId
left join categories c on c.category_id=o.category_id
group by c.category_name, t.TransactionTypeId

### Top Organization by Category

In [ ]:
%%sql

with t2 as
(select t1.*,
       row_number() over(partition by t1.category_name order by t1.tr_count desc) as rownum
from
(select c.category_name,
       o.organization_name,
       count(Id) as tr_count
from transactions t
left join organizations o on o.organization_id=t.OrgId
left join categories c on c.category_id=o.category_id
group by c.category_name, o.organization_name ) t1 )

select category_name,
       organization_name,
       tr_count
from t2
where rownum=1